<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/4_2_Generalized_Linear_Models_(GLM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part IV. Generalized Linear Models, Bayesian Methods, and Latent Variables



## Generalized Linear Models (GLM)

## Bayesian Linear Regression

## Bayesian Logistic Regression

## Latent Variable Models: EM Algorithm and GMM

### Экспоненциальное семейство распределений и мотивация обобщённых линейных моделей

Классическая линейная регрессия с нормальным распределением ошибок — краеугольный камень статистического моделирования. Однако её прямое применение ограничено: во многих реальных задачах отклик $Y$ не является непрерывной величиной с постоянной дисперсией. Он может быть бинарным (успех/неудача), счётчиком (число событий), временем до отказа или долей. Попытки загнать такие данные в прокрустово ложе нормальной линейной модели — либо путём преобразования отклика, либо игнорированием природы данных — ведут к неэффективным, смещённым или попросту некорректным выводам. Обобщённые линейные модели (GLM) изящно решают эту проблему, единообразно описывая широкий спектр распределений отклика и связывая его математическое ожидание с линейным предиктором через произвольную монотонную функцию связи.

#### 1. Мотивация: ограничения классической линейной регрессии

Классическая линейная модель предполагает, что отклик $Y$ при заданных признаках $\mathbf{x}$ распределён нормально с постоянной дисперсией $\sigma^2$, а его среднее линейно зависит от $\mathbf{x}$:

$$
Y \mid \mathbf{x} \;\sim\; \mathcal{N}(\mathbf{x}^T\boldsymbol{\beta}, \sigma^2), \qquad \mathbb{E}[Y \mid \mathbf{x}] = \mathbf{x}^T\boldsymbol{\beta}.
$$

Эта модель оптимальна, когда отклик действительно нормален, но становится неадекватной во многих практических ситуациях:

- **Бинарный отклик** (0/1). Предсказание вероятности успеха не может быть линейной функцией признаков, так как вероятность должна лежать в $[0,1]$, а линейный предиктор способен выходить далеко за эти границы. К тому же бинарные данные гетероскедастичны: дисперсия зависит от среднего ($\operatorname{Var}(Y) = p(1-p)$).
- **Счётные данные** (число звонков, количество отказов). Такие величины неотрицательны и целочисленны, а их дисперсия часто растёт со средним (пуассоновское поведение).
- **Времена жизни** — положительные, часто с длинным правым хвостом (экспоненциальное, гамма-распределение).
- **Доли, проценты** — ограничены интервалом $[0,1]$.

Классические подходы к нормализации (логарифмирование, извлечение корня) могут помочь, но они трансформируют не только среднее, но и дисперсию, а также искажают интерпретацию параметров на исходной шкале. GLM предлагают альтернативу: вместо преобразования данных мы выбираем подходящее распределение для $Y$ и моделируем преобразованное среднее как линейную функцию предикторов.

#### 2. Экспоненциальное семейство распределений

Фундаментом GLM служит экспоненциальное семейство распределений. Случайная величина $Y$ принадлежит экспоненциальному семейству, если её плотность (или вероятностная масса) может быть записана в канонической форме

$$
p(y \mid \theta, \phi) = \exp\!\left( \frac{y\theta - b(\theta)}{a(\phi)} + c(y, \phi) \right), \tag{1}
$$

где
- $\theta$ — **естественный (канонический) параметр**, определяющий распределение,
- $\phi$ — **параметр масштаба** (дисперсионный параметр), часто известный или оцениваемый отдельно,
- $a(\phi)$ — положительная функция, обычно $a(\phi) = \phi / w$ с известным весом $w$,
- $b(\theta)$ — функция, дифференцируемая нужное число раз; её производные дают моменты распределения,
- $c(y, \phi)$ — функция, не зависящая от $\theta$ и обеспечивающая нормировку.

В это семейство входит подавляющее большинство практически важных распределений. Приведём основные примеры, записывая их в каноническом виде.

**Нормальное распределение** $Y \sim \mathcal{N}(\mu, \sigma^2)$. Фиксируем $\sigma^2$ (параметр масштаба) и запишем плотность:

$$
p(y \mid \mu) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(y-\mu)^2}{2\sigma^2}\right)
= \exp\!\left( \frac{y\mu - \mu^2/2}{\sigma^2} - \frac{y^2}{2\sigma^2} - \frac{1}{2}\log(2\pi\sigma^2) \right).
$$

Здесь $\theta = \mu$, $b(\theta) = \theta^2/2$, $a(\phi) = \sigma^2$, $\phi = \sigma^2$, $c(y, \phi) = -\frac{y^2}{2\sigma^2} - \frac{1}{2}\log(2\pi\sigma^2)$.

**Биномиальное распределение** (для $m$ испытаний с вероятностью успеха $p$). Вероятностная масса:

$$
P(Y=y) = \binom{m}{y} p^y (1-p)^{m-y}
= \exp\!\left( y \log\frac{p}{1-p} + m \log(1-p) + \log\binom{m}{y} \right).
$$

Положим $\theta = \log\frac{p}{1-p}$ (логит), тогда $p = \frac{e^\theta}{1+e^\theta}$, $\log(1-p) = -\log(1+e^\theta)$. Получаем

$$
P(Y=y) = \exp\!\left( \frac{y\theta - m \log(1+e^\theta)}{1} + \log\binom{m}{y} \right).
$$

Здесь $a(\phi) = 1$, $b(\theta) = m \log(1+e^\theta)$, $c(y) = \log\binom{m}{y}$.

**Пуассоновское распределение** $Y \sim \operatorname{Poisson}(\lambda)$:

$$
P(Y=y) = \frac{\lambda^y e^{-\lambda}}{y!} = \exp\!\bigl( y\log\lambda - \lambda - \log y! \bigr).
$$

Канонический параметр $\theta = \log\lambda$, $b(\theta) = e^\theta$, $a(\phi)=1$, $c(y)=-\log y!$.

**Гамма-распределение** с фиксированным параметром формы $\nu$ и средним $\mu$:

$$
p(y) = \frac{1}{\Gamma(\nu)} \left(\frac{\nu y}{\mu}\right)^\nu \exp\!\left(-\frac{\nu y}{\mu}\right) \frac{1}{y}
= \exp\!\left( \nu\left( -\frac{y}{\mu} - \log\mu \right) + (\nu-1)\log y - \log\Gamma(\nu) + \nu\log\nu \right).
$$

Канонический параметр $\theta = -1/\mu$, $b(\theta) = -\log(-\theta)$, $a(\phi) = 1/\nu$, $\phi = 1/\nu$.

#### 3. Свойства экспоненциального семейства

Элегантность экспоненциального семейства — в том, что моменты распределения выражаются через производные $b(\theta)$. Логарифмируя (1) и дифференцируя по $\theta$, получаем

$$
\frac{\partial}{\partial\theta} \log p(y \mid \theta, \phi) = \frac{y - b'(\theta)}{a(\phi)}.
$$

Математическое ожидание производной логарифма правдоподобия равно нулю (при условии регулярности), откуда

$$
\mathbb{E}[Y] = b'(\theta). \tag{2}
$$

Аналогично, вторая производная даёт

$$
\frac{\partial^2}{\partial\theta^2} \log p(y \mid \theta, \phi) = -\frac{b''(\theta)}{a(\phi)},
$$

и поскольку $\mathbb{E}\left[-\frac{\partial^2 \log p}{\partial\theta^2}\right] = \mathbb{E}\left[\left(\frac{\partial \log p}{\partial\theta}\right)^2\right]$, получаем

$$
\operatorname{Var}(Y) = b''(\theta)\, a(\phi). \tag{3}
$$

Функция $b(\theta)$ полностью определяет зависимость между средним и дисперсией: дисперсия является функцией среднего через $V(\mu) = b''((b')^{-1}(\mu))$, называемую **функцией дисперсии**. Для нормального распределения $V(\mu) = 1$, для пуассоновского $V(\mu) = \mu$, для биномиального $V(\mu) = \mu(1-\mu/m)$, для гамма $V(\mu) = \mu^2$. Это свойство GLM — правильное моделирование зависимости дисперсии от среднего — принципиально для получения эффективных оценок и корректных доверительных интервалов.

#### 4. Компоненты обобщённой линейной модели

GLM унифицирует линейные модели через три составляющие.

1. **Случайный компонент**: отклик $Y$ распределён по некоторому закону из экспоненциального семейства с параметрами $\theta$ и $\phi$.
2. **Систематический компонент**: линейный предиктор $\eta = \mathbf{x}^T\boldsymbol{\beta}$, где $\mathbf{x}$ — вектор признаков (включая, возможно, константу), $\boldsymbol{\beta}$ — вектор коэффициентов.
3. **Функция связи $g$**: строго монотонная и дифференцируемая функция, связывающая среднее $\mu = \mathbb{E}[Y \mid \mathbf{x}]$ с линейным предиктором: $g(\mu) = \eta$.

Если $g$ является обратной функцией к $b'$, т.е. $g = (b')^{-1}$, так что $\theta = \eta$, говорят о **канонической функции связи**. В этом случае естественный параметр $\theta$ прямо равен линейному предиктору, что значительно упрощает математику и гарантирует выпуклость логарифмической функции правдоподобия.

Для нормальной модели каноническая связь — тождественная $g(\mu) = \mu$ (обычная линейная регрессия). Для биномиальной — логит $g(\mu) = \log\frac{\mu}{1-\mu}$ (логистическая регрессия). Для пуассоновской — логарифмическая $g(\mu) = \log \mu$ (лог-линейная модель). Для гамма — обратная $g(\mu) = -1/\mu$.

#### 5. Преимущества GLM перед преобразованием отклика

Традиционный подход к «ненормальным» данным — подобрать преобразование $Y^* = h(Y)$, после которого $Y^*$ становится приблизительно нормальным, и применить обычную линейную регрессию. Несмотря на простоту, такой метод имеет серьёзные недостатки. Во-первых, преобразование меняет не только форму распределения, но и характер связи между $Y$ и предикторами: если $\mathbb{E}[h(Y)]$ линейно зависит от $\mathbf{x}$, это не означает линейности исходного среднего $\mathbb{E}[Y]$. Во-вторых, дисперсия $h(Y)$ может остаться непостоянной. В-третьих, при дискретных данных (нули, единицы) преобразование не может сделать распределение нормальным.

GLM решает эти проблемы, явно задавая распределение $Y$ и моделируя среднее на исходной шкале через функцию связи. Это даёт корректную оценку стандартных ошибок, состоятельные и эффективные оценки параметров, и естественную интерпретацию коэффициентов. Например, в логистической регрессии коэффициенты интерпретируются как логарифмы отношений шансов, в пуассоновской — как логарифмы относительного риска.

> **Углублённый взгляд: вывод канонической связи и уравнений правдоподобия**  
> Рассмотрим обучение GLM методом максимального правдоподобия. Логарифмическое правдоподобие для $n$ независимых наблюдений имеет вид
> $$
> \ell(\boldsymbol{\beta}) = \sum_{i=1}^n \left( \frac{y_i \theta_i - b(\theta_i)}{a_i(\phi)} + c(y_i, \phi) \right),
> $$
> где $\theta_i$ зависит от $\boldsymbol{\beta}$ через $\mu_i = b'(\theta_i)$ и $g(\mu_i) = \mathbf{x}_i^T\boldsymbol{\beta}$. Градиент $\ell$ по $\boldsymbol{\beta}$:
> $$
> \frac{\partial \ell}{\partial \beta_j} = \sum_{i=1}^n \frac{y_i - \mu_i}{a_i(\phi)} \cdot \frac{\partial \theta_i}{\partial \beta_j}.
> $$
> По цепному правилу $\frac{\partial \theta_i}{\partial \beta_j} = \frac{d\theta_i}{d\mu_i} \frac{d\mu_i}{d\eta_i} x_{ij}$, где $\frac{d\theta_i}{d\mu_i} = 1 / b''(\theta_i) = 1 / V(\mu_i)$. Следовательно,
> $$
> \frac{\partial \ell}{\partial \beta_j} = \sum_{i=1}^n \frac{y_i - \mu_i}{a_i(\phi) V(\mu_i) g'(\mu_i)} x_{ij}. \tag{4}
> $$
> В случае канонической связи $\theta_i = \eta_i$, поэтому $\frac{d\theta_i}{d\eta_i}=1$, и градиент принимает особенно простую форму:
> $$
> \frac{\partial \ell}{\partial \beta_j} = \sum_{i=1}^n \frac{y_i - \mu_i}{a_i(\phi)} x_{ij}.
> $$
> Приравнивая его к нулю, получаем **уравнения правдоподобия**:
> $$
> \mathbf{X}^T (\mathbf{y} - \boldsymbol{\mu}) = \mathbf{0}, \tag{5}
> $$
> где $\mathbf{y} = (y_1,\dots,y_n)^T$, $\boldsymbol{\mu} = (\mu_1,\dots,\mu_n)^T$, а $\mathbf{X}$ — матрица плана. Это означает, что при канонической связи остатки ортогональны подпространству, натянутому на столбцы матрицы признаков — точный аналог нормальных уравнений классической линейной регрессии. Именно каноническая связь гарантирует выпуклость функции правдоподобия и часто упрощает вычисления.

Таким образом, GLM не просто расширяет линейную регрессию на другие распределения, но и сохраняет её концептуальную стройность, добавляя гибкость в моделировании среднего и дисперсии. В следующем разделе мы подробно рассмотрим оценку параметров GLM методом максимального правдоподобия с использованием итеративного взвешенного метода наименьших квадратов.

---

#### Вопросы для самопроверки

**Для начинающих**
1. Что такое экспоненциальное семейство распределений? Приведите три примера распределений, входящих в него, с указанием естественного параметра $\theta$.
2. Для чего в обобщённых линейных моделях служит функция связи? Приведите пример функции связи, отличной от канонической, для биномиального отклика.
3. Чем GLM принципиально отличается от классической линейной регрессии с предварительным преобразованием отклика?
4. Дайте определение канонической функции связи. Почему она удобна?
5. Как в GLM моделируется зависимость дисперсии отклика от его среднего? Приведите пример для пуассоновского распределения.

**Для профессионалов**
1. Выведите формулы $\mathbb{E}[Y] = b'(\theta)$ и $\operatorname{Var}(Y) = b''(\theta) a(\phi)$ из общей формы плотности экспоненциального семейства.
2. Покажите, что для биномиального распределения с числом испытаний $m$ канонической связью является логит-функция $\theta = \log\frac{p}{1-p}$, а $b(\theta) = m\log(1+e^\theta)$.
3. Запишите функцию логарифмического правдоподобия для GLM и выведите выражение для её градиента по $\boldsymbol{\beta}$ в общем случае (4). При каком условии градиент принимает вид $\mathbf{X}^T(\mathbf{y} - \boldsymbol{\mu})$?
4. Объясните, почему для гамма-распределения с фиксированным параметром формы канонической связью является обратная функция $g(\mu) = -1/\mu$. Какова при этом функция $b(\theta)$?

### Оценка параметров GLM: метод максимального правдоподобия и итеративно перевзвешенный метод наименьших квадратов

Определив структуру обобщённой линейной модели — экспоненциальное семейство, линейный предиктор и функцию связи, — мы переходим к центральной задаче: оцениванию вектора коэффициентов $\boldsymbol{\beta}$ по обучающей выборке. В отличие от классической линейной регрессии, где явная формула МНК даёт готовое решение, в GLM уравнения правдоподобия нелинейны и требуют итеративной техники. Основным инструментом служит итеративно перевзвешенный метод наименьших квадратов (IRLS), объединяющий идейную простоту взвешенной линейной регрессии со скоростью сходимости метода Ньютона.

#### 1. Логарифмическое правдоподобие для GLM

Пусть имеется $n$ независимых наблюдений $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$, где отклик $y_i$ подчиняется распределению из экспоненциального семейства с каноническим параметром $\theta_i$, связанным со средним $\mu_i = \mathbb{E}[Y_i]$ посредством $\mu_i = b'(\theta_i)$. Линейный предиктор $\eta_i = \mathbf{x}_i^T \boldsymbol{\beta}$ связан со средним через функцию связи $g(\mu_i) = \eta_i$. Логарифмическое правдоподобие всей выборки, с точностью до слагаемых, не зависящих от $\boldsymbol{\beta}$, имеет вид

$$
\ell(\boldsymbol{\beta}) = \sum_{i=1}^n \frac{y_i \theta_i - b(\theta_i)}{a(\phi)}.
\tag{1}
$$

Здесь $a(\phi)$ обычно представима как $\phi / w_i$, где $w_i$ — известные веса наблюдений (например, $w_i = 1$). Зависимость $\ell$ от $\boldsymbol{\beta}$ полностью определяется зависимостью $\theta_i = \theta(\mu_i)$ и $\mu_i = g^{-1}(\eta_i) = g^{-1}(\mathbf{x}_i^T \boldsymbol{\beta})$.

#### 2. Уравнения правдоподобия (score equations)

Максимизация $\ell(\boldsymbol{\beta})$ приводит к системе уравнений правдоподобия $\nabla_{\boldsymbol{\beta}} \ell(\boldsymbol{\beta}) = \mathbf{0}$. Вычислим частную производную по $\beta_j$ с помощью цепного правила:

$$
\frac{\partial \ell}{\partial \beta_j} =
\sum_{i=1}^n \frac{\partial \ell}{\partial \theta_i} \cdot
\frac{\partial \theta_i}{\partial \mu_i} \cdot
\frac{\partial \mu_i}{\partial \eta_i} \cdot
\frac{\partial \eta_i}{\partial \beta_j}.
$$

Из (1) имеем $\partial \ell / \partial \theta_i = \bigl(y_i - b'(\theta_i)\bigr) / a(\phi) = (y_i - \mu_i) / a(\phi)$.
Далее, $\partial \theta_i / \partial \mu_i = 1 / b''(\theta_i) = 1 / V(\mu_i)$, где $V(\mu) = b''\bigl((b')^{-1}(\mu)\bigr)$ — функция дисперсии.
По определению функции связи $g(\mu_i) = \eta_i$, поэтому $\partial \mu_i / \partial \eta_i = 1 / g'(\mu_i)$.
Наконец, $\partial \eta_i / \partial \beta_j = x_{ij}$.

Собирая множители, получаем общий вид уравнения правдоподобия:

$$
\frac{\partial \ell}{\partial \beta_j} =
\sum_{i=1}^n \frac{(y_i - \mu_i)}{a(\phi)\, V(\mu_i)\, g'(\mu_i)} \, x_{ij} = 0, \qquad j = 1,\dots,p.
\tag{2}
$$

В векторно-матричной форме эту систему можно записать как

$$
\mathbf{X}^T \mathbf{D}^{-1} (\mathbf{y} - \boldsymbol{\mu}) = \mathbf{0},
$$

где $\mathbf{D} = \operatorname{diag}\bigl( a(\phi)\, V(\mu_i)\, g'(\mu_i) \bigr)$.

Особенно изящный вид уравнения принимают при **канонической функции связи**, когда $\theta_i = \eta_i$, т.е. $g = (b')^{-1}$. В этом случае $g'(\mu_i) = 1 / V(\mu_i)$, и произведение $V(\mu_i) g'(\mu_i) = 1$, а значит, (2) редуцируется до

$$
\sum_{i=1}^n \frac{y_i - \mu_i}{a(\phi)} x_{ij} = 0,
\quad\text{или}\quad
\mathbf{X}^T (\mathbf{y} - \boldsymbol{\mu}) = \mathbf{0}.
\tag{3}
$$

Таким образом, при канонической связи остатки должны быть ортогональны пространству признаков — точный аналог нормальных уравнений классического метода наименьших квадратов. Эта простота является одной из причин популярности канонических связей.

#### 3. Итеративно перевзвешенный метод наименьших квадратов (IRLS)

Уравнения (2) нелинейны по $\boldsymbol{\beta}$, так как $\mu_i$ и $g'(\mu_i)$ сами зависят от $\boldsymbol{\beta}$. Для их решения применяют итеративную процедуру, на каждом шаге аппроксимируя задачу взвешенной линейной регрессией. Введём **рабочую переменную** (adjusted dependent variable)

$$
z_i = \eta_i + (y_i - \mu_i) \frac{d\eta_i}{d\mu_i}
      = \mathbf{x}_i^T \boldsymbol{\beta} + (y_i - \mu_i)\, g'(\mu_i).
\tag{4}
$$

Линеаризуя $g(y_i)$ вокруг $\mu_i$, $z_i$ можно интерпретировать как «наблюдаемое значение» линейного предиктора. Далее определим **веса**

$$
w_i = \left( \frac{d\mu_i}{d\eta_i} \right)^2 \frac{1}{\operatorname{Var}(Y_i)}
     = \frac{1}{a(\phi)\, V(\mu_i)\, \bigl(g'(\mu_i)\bigr)^2},
\tag{5}
$$

которые обратно пропорциональны дисперсии $z_i$. С этими обозначениями уравнение (2) можно переписать как

$$
\sum_{i=1}^n w_i (z_i - \mathbf{x}_i^T \boldsymbol{\beta}) \, x_{ij} = 0,
$$

что в точности является условием минимума взвешенной суммы квадратов $\sum_i w_i (z_i - \mathbf{x}_i^T \boldsymbol{\beta})^2$. Таким образом, если бы величины $\mu_i$ (и, следовательно, $w_i, z_i$) были фиксированы, оценка $\boldsymbol{\beta}$ находилась бы явно:

$$
\hat{\boldsymbol{\beta}} = (\mathbf{X}^T \mathbf{W} \mathbf{X})^{-1} \mathbf{X}^T \mathbf{W} \mathbf{z},
$$

где $\mathbf{W} = \operatorname{diag}(w_1,\dots,w_n)$, $\mathbf{z} = (z_1,\dots,z_n)^T$.

IRLS итеративно чередует вычисление $\mu_i$, $w_i$, $z_i$ по текущему приближению $\boldsymbol{\beta}^{(t)}$ и решение взвешенной линейной регрессии для получения $\boldsymbol{\beta}^{(t+1)}$. Формально один шаг выглядит так:

$$
\boldsymbol{\beta}^{(t+1)} = \bigl(\mathbf{X}^T \mathbf{W}^{(t)} \mathbf{X}\bigr)^{-1} \mathbf{X}^T \mathbf{W}^{(t)} \mathbf{z}^{(t)},
\tag{6}
$$

где $\mathbf{W}^{(t)}$ и $\mathbf{z}^{(t)}$ вычислены с использованием $\boldsymbol{\beta}^{(t)}$. Процесс повторяется до сходимости.

Этот алгоритм совпадает с **методом Фишера** (Fisher scoring) — вариантом метода Ньютона, в котором Гессиан логарифмического правдоподобия заменяется на своё математическое ожидание (информационную матрицу Фишера). Благодаря этому IRLS обладает надёжной сходимостью: информационная матрица положительно определена, что гарантирует возрастание правдоподобия на каждой итерации при достаточно хорошем начальном приближении.

#### 4. Сходимость и начальное приближение

Вблизи максимума правдоподобия IRLS демонстрирует квадратичную сходимость, типичную для методов ньютоновского типа. Обычно требуется всего 5–10 итераций для достижения высокой точности. Начальное приближение $\boldsymbol{\beta}^{(0)}$ можно выбрать разными способами:

- Для нормальной линейной регрессии — МНК-оценка из модели с $g(\mu)=\mu$.
- Для логистической регрессии — нулевой вектор, либо грубая оценка по частотам классов.
- Для лог-линейной (пуассоновской) модели — логарифмы выборочных средних.

Важно помнить, что в случае логистической регрессии при полной (или квази-полной) разделимости классов в обучающей выборке максимум правдоподобия достигается на бесконечности, и IRLS не сходится: веса некоторых объектов стремятся к нулю, а коэффициенты расходятся. Обнаружение такой ситуации и применение регуляризации — необходимые меры предосторожности.

#### 5. Асимптотические свойства оценок

При стандартных условиях регулярности оценка максимального правдоподобия $\hat{\boldsymbol{\beta}}$ является состоятельной и асимптотически нормальной:

$$
\hat{\boldsymbol{\beta}} \;\dot\sim\; \mathcal{N}\Bigl( \boldsymbol{\beta},\; \mathcal{I}(\boldsymbol{\beta})^{-1} \Bigr),
$$

где $\mathcal{I}(\boldsymbol{\beta}) = \mathbb{E}\bigl[-\nabla_{\boldsymbol{\beta}}^2 \ell(\boldsymbol{\beta})\bigr]$ — информационная матрица Фишера. Используя выражение для ожидаемого Гессиана, получаем

$$
\mathcal{I}(\boldsymbol{\beta}) = \mathbf{X}^T \mathbf{W} \mathbf{X} \,\big/\, \phi,
$$

если $a(\phi) = \phi$ (или $a(\phi) = \phi / w_i$). Таким образом, ковариационная матрица оценок приближённо равна

$$
\widehat{\operatorname{Cov}}(\hat{\boldsymbol{\beta}}) = \hat{\phi} \bigl( \mathbf{X}^T \hat{\mathbf{W}} \mathbf{X} \bigr)^{-1},
\tag{7}
$$

где $\hat{\mathbf{W}}$ — матрица весов, вычисленная в точке $\hat{\boldsymbol{\beta}}$, а $\hat{\phi}$ — оценка параметра масштаба $\phi$ (если он не известен заранее). Для моделей с фиксированным $\phi = 1$ (логистическая, пуассоновская) параметр масштаба не оценивается.

Стандартные ошибки коэффициентов, получаемые как квадратные корни из диагональных элементов матрицы (7), служат основой для построения доверительных интервалов и проверки гипотез (тест Вальда). Наряду с тестом отношения правдоподобия и тестом на основе скоров, они образуют полный арсенал статистического вывода в GLM.

> **Углублённый взгляд: вывод IRLS через метод Ньютона–Рафсона**  
> Метод Ньютона для поиска максимума $\ell(\boldsymbol{\beta})$ обновляет параметры по правилу
> $$
> \boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} - \bigl[ \mathbf{H}^{(t)} \bigr]^{-1} \nabla \ell(\boldsymbol{\beta}^{(t)}),
> $$
> где $\mathbf{H}$ — матрица Гессе логарифмического правдоподобия. Используя (2), можно показать, что Гессиан состоит из двух слагаемых:
> $$
> \mathbf{H} = - \mathbf{X}^T \mathbf{W} \mathbf{X} + \mathbf{R},
> $$
> где $\mathbf{R}$ — матрица, зависящая от остатков $(y_i - \mu_i)$ и обращающаяся в нуль при канонической связи (или при равенстве наблюдаемых и ожидаемых значений). В методе Фишера, который и реализует IRLS, поправочный член $\mathbf{R}$ отбрасывается, и используется **ожидаемый Гессиан** $-\mathbf{X}^T \mathbf{W} \mathbf{X}$. Подставляя градиент $\nabla \ell = \mathbf{X}^T \mathbf{W} (\mathbf{z} - \boldsymbol{\eta})$ (с точностью до констант), получаем в точности формулу (6). При канонической связи $\mathbf{R} \equiv 0$, так что наблюдаемый и ожидаемый Гессианы совпадают, и IRLS становится классическим методом Ньютона. Это объясняет быструю сходимость канонических GLM.  
> Таким образом, IRLS — не просто эвристика, а строгий ньютоновский метод, адаптированный к структуре GLM и гарантирующий монотонное возрастание правдоподобия.

---

#### Вопросы для самопроверки

**Для начинающих**
1. Что означает аббревиатура IRLS и почему метод назван «перевзвешенным»?
2. Как выглядят уравнения правдоподобия для логистической регрессии (каноническая связь) в терминах $\mathbf{X}$, $\mathbf{y}$ и $\boldsymbol{\mu}$?
3. Зачем в IRLS вводятся рабочая переменная $z_i$ и веса $w_i$? Какая задача решается на каждой итерации?
4. Как получить начальное приближение для коэффициентов в логистической регрессии?
5. В чём преимущество канонической связи с точки зрения сходимости IRLS?

**Для профессионалов**
1. Выведите итерационную формулу IRLS (6) непосредственно из метода Ньютона–Рафсона для логарифмического правдоподобия GLM. Покажите, какое слагаемое отбрасывается при переходе к ожидаемому Гессиану.
2. Докажите, что при канонической функции связи Гессиан логарифмического правдоподобия отрицательно полуопределён, а значит, задача максимизации выпукла.
3. Объясните, как из матрицы $\mathbf{W}$ в последней итерации IRLS получить ковариационную матрицу оценок $\hat{\boldsymbol{\beta}}$. Как оценивается параметр масштаба $\phi$ для гауссовой и гамма-моделей?
4. Сформулируйте условие сходимости IRLS для логистической регрессии. Почему при линейно разделимой обучающей выборке алгоритм расходится, и какие методы решения этой проблемы существуют?

### Частные случаи: логистическая регрессия, пуассоновская регрессия, гамма-регрессия

Конкретные представители семейства GLM получаются выбором распределения отклика и функции связи. Каждая модель привносит свою интерпретацию параметров, свои особенности оценивания и свои области применения. Ниже мы подробно рассмотрим три наиболее востребованные модели: логистическую регрессию для бинарных и биномиальных данных, пуассоновскую регрессию для счётчиков и гамма-регрессию для положительных непрерывных величин.

#### 1. Логистическая регрессия (биномиальный отклик)

Пусть $Y_i$ — число успехов в $m_i$ независимых испытаниях с вероятностью успеха $p_i$, так что $Y_i \sim \operatorname{Binomial}(m_i, p_i)$. Среднее $\mu_i = m_i p_i$, дисперсия $\operatorname{Var}(Y_i) = m_i p_i(1-p_i)$. Канонической функцией связи служит логит-преобразование:

$$
g(p_i) = \log\frac{p_i}{1-p_i} = \eta_i = \mathbf{x}_i^T \boldsymbol{\beta}.
\tag{1}
$$

Такая параметризация автоматически гарантирует $p_i \in (0,1)$, а линейный предиктор может принимать любые вещественные значения. Обратная связь — логистическая функция:

$$
p_i = \frac{1}{1 + \exp(-\eta_i)}.
$$

Для реализации IRLS конкретизируем компоненты. Функция дисперсии $V(\mu_i) = \mu_i(1 - \mu_i/m_i)$. Производная обратной связи $\frac{d\mu_i}{d\eta_i} = m_i p_i(1-p_i)$. Тогда веса и рабочая переменная для наблюдения $i$ принимают вид

$$
w_i = \left(\frac{d\mu_i}{d\eta_i}\right)^2 \frac{1}{\operatorname{Var}(Y_i)} = m_i \hat{p}_i (1-\hat{p}_i),
$$

$$
z_i = \eta_i + \frac{y_i - m_i \hat{p}_i}{m_i \hat{p}_i (1-\hat{p}_i)},
$$

где $\hat{p}_i$ вычислено по текущему $\boldsymbol{\beta}$. При $m_i=1$ (бинарная логистическая регрессия) веса упрощаются до $w_i = \hat{p}_i(1-\hat{p}_i)$.

Интерпретация коэффициентов логистической регрессии — одна из её самых сильных сторон. При увеличении $j$-го признака на единицу (при фиксированных остальных) логарифм шансов (log-odds) успеха изменяется на $\beta_j$:

$$
\log\frac{p(\mathbf{x} + \mathbf{e}_j)}{1-p(\mathbf{x} + \mathbf{e}_j)} - \log\frac{p(\mathbf{x})}{1-p(\mathbf{x})} = \beta_j.
$$

Следовательно, экспонента коэффициента $e^{\beta_j}$ является **отношением шансов** (odds ratio): шансы успеха умножаются на $e^{\beta_j}$ при единичном приращении $x_j$. Это делает модель незаменимой в эпидемиологии, медицине и социальных науках, где оценивают влияние факторов риска.

#### 2. Пуассоновская регрессия

Пуассоновская регрессия применяется к счётным данным: количество событий за фиксированный промежуток времени или в фиксированной области. Модель предполагает $Y_i \sim \operatorname{Poisson}(\lambda_i)$ со средним $\mu_i = \lambda_i > 0$ и дисперсией $\operatorname{Var}(Y_i) = \lambda_i$. Каноническая связь — логарифмическая:

$$
\log \lambda_i = \eta_i = \mathbf{x}_i^T \boldsymbol{\beta},
\tag{2}
$$

откуда $\lambda_i = \exp(\mathbf{x}_i^T \boldsymbol{\beta})$, что автоматически обеспечивает положительность среднего. Функция дисперсии $V(\mu) = \mu$, производная обратной связи $\frac{d\mu}{d\eta} = \mu$. Веса IRLS: $w_i = \hat{\lambda}_i$, рабочая переменная:

$$
z_i = \eta_i + \frac{y_i - \hat{\lambda}_i}{\hat{\lambda}_i}.
$$

Интерпретация коэффициентов: при увеличении $x_j$ на единицу среднее число событий умножается на $e^{\beta_j}$ (относительный риск). Например, если $\beta_j = 0.2$, то $e^{0.2} \approx 1.22$, что соответствует увеличению ожидаемого числа событий на 22%.

Фундаментальная проблема пуассоновской регрессии — **передисперсия** (overdispersion): реальная дисперсия часто превышает среднее, тогда как модель жёстко связывает их равенством $\operatorname{Var}(Y) = \mu$. Передисперсия может возникать из-за ненаблюдаемой гетерогенности, кластеризации или пропущенных предикторов. Её наличие приводит к заниженным стандартным ошибкам и ложнозначимым выводам. Простейшее решение — **квази-пуассоновская модель**, в которой дисперсия полагается пропорциональной среднему: $\operatorname{Var}(Y) = \phi \mu$, а параметр масштаба $\phi$ оценивается по остаткам (обычно $\hat{\phi} = \frac{1}{n-p}\sum \frac{(y_i-\hat{\mu}_i)^2}{\hat{\mu}_i}$). Более гибкий подход — отрицательная биномиальная регрессия, вводящая дополнительный параметр для управления дисперсией.

#### 3. Гамма-регрессия

Гамма-распределение является естественной моделью для положительных непрерывных величин с постоянным коэффициентом вариации: время обслуживания, суммы страховых выплат, продолжительность безотказной работы. Плотность гамма-распределения с параметром формы $\nu$ (фиксированным) и средним $\mu$:

$$
p(y \mid \mu, \nu) = \frac{1}{\Gamma(\nu)} \left(\frac{\nu}{\mu}\right)^\nu y^{\nu-1} \exp\!\left(-\frac{\nu y}{\mu}\right), \quad y > 0.
$$

Дисперсия $\operatorname{Var}(Y) = \mu^2 / \nu$. Каноническая функция связи для гамма-распределения — обратная: $\eta = -1/\mu$. Однако на практике чаще применяют логарифмическую связь $\log \mu = \eta$, так как она гарантирует положительность среднего и даёт простую интерпретацию: при единичном изменении $x_j$ среднее умножается на $e^{\beta_j}$. Кроме того, обратная связь может давать отрицательные предсказания среднего при некоторых значениях предикторов.

Стоит отметить различие между гамма-регрессией и логарифмированием отклика с последующей линейной регрессией. Лог-нормальная модель предполагает, что $\log Y$ нормален, что подразумевает иное поведение дисперсии и хвостов. Гамма-регрессия моделирует среднее напрямую, и оценки коэффициентов интерпретируются как мультипликативные эффекты на исходной шкале, а не на шкале логарифма геометрического среднего. Выбор между ними диктуется природой данных и удобством интерпретации.

#### 4. Сравнение моделей и выбор функции связи

Каноническая связь не всегда является наилучшим выбором. Хотя она обеспечивает выпуклость логарифмического правдоподобия и упрощает вычисления, содержательные соображения могут диктовать другую функцию связи. Типичные альтернативы:

- Для биномиального отклика: **пробит-связь** $\Phi^{-1}(p) = \eta$ (где $\Phi$ — функция стандартного нормального распределения) и **комплементарная log-log связь** $\log(-\log(1-p)) = \eta$, удобная при анализе выживаемости.
- Для пуассоновского отклика: **квадратный корень** $\sqrt{\mu} = \eta$, стабилизирующий дисперсию.
- Для гамма-отклика: **логарифмическая связь** $\log\mu = \eta$, как уже упоминалось, наиболее популярна.

Диагностика адекватности выбранной модели опирается на анализ остатков. Два основных типа остатков в GLM:

- **Остатки Пирсона**: $r_{P,i} = \frac{y_i - \hat{\mu}_i}{\sqrt{\widehat{\operatorname{Var}}(Y_i)}}$. Их сумма квадратов даёт обобщённую статистику хи-квадрат.
- **Девиансные остатки**: $r_{D,i} = \operatorname{sign}(y_i - \hat{\mu}_i) \sqrt{d_i}$, где $d_i$ — вклад $i$-го наблюдения в девианс (см. ниже). Девиансные остатки более симметричны для несимметричных распределений.

#### 5. Оценка качества и выбор модели

Центральную роль в GLM играет **девианс** (deviance) — аналог суммы квадратов остатков в линейной регрессии. Пусть $\ell(\hat{\boldsymbol{\mu}})$ — логарифмическое правдоподобие модели, а $\ell_{\text{sat}}$ — логарифмическое правдоподобие насыщенной модели, в которой для каждого наблюдения подбирается отдельный параметр $\tilde{\mu}_i = y_i$. Девианс определяется как

$$
D = 2\bigl( \ell_{\text{sat}} - \ell(\hat{\boldsymbol{\mu}}) \bigr)
   = \sum_{i=1}^n d_i,
$$

где $d_i$ — вклад $i$-го наблюдения. Девианс измеряет различие между подогнанной моделью и «идеальной» моделью, точно воспроизводящей данные. Чем меньше девианс, тем лучше модель соответствует данным. Для сравнения вложенных моделей используют разность девиансов, которая при нулевой гипотезе (более простая модель верна) асимптотически распределена как $\chi^2$ с числом степеней свободы, равным разности числа параметров.

**Информационные критерии AIC и BIC**, основанные на девиансе и штрафе за число параметров, служат универсальным инструментом выбора модели:

$$
\operatorname{AIC} = D + 2p,\qquad
\operatorname{BIC} = D + p \log n,
$$

где $p$ — число оцениваемых параметров, $n$ — объём выборки. Меньшие значения критериев указывают на более предпочтительную модель, балансирующую качество подгонки и сложность.

> **Углублённый взгляд: вывод девианса для нормальной и пуассоновской моделей**  
> Продемонстрируем, как вычисляется девианс для конкретных распределений.  
> *Нормальная линейная регрессия.* Пусть $Y_i \sim \mathcal{N}(\mu_i, \sigma^2)$. Логарифмическое правдоподобие с точностью до констант: $\ell(\boldsymbol{\mu}) = -\frac{1}{2\sigma^2}\sum_{i=1}^n (y_i - \mu_i)^2$. Насыщенная модель полагает $\tilde{\mu}_i = y_i$, тогда $\ell_{\text{sat}} = 0$ (константы опущены). Отсюда  
> $$
> D = 2\bigl(0 - (-\frac{1}{2\sigma^2}\sum (y_i - \hat{\mu}_i)^2)\bigr) = \frac{1}{\sigma^2} \sum_{i=1}^n (y_i - \hat{\mu}_i)^2.
> $$
> Таким образом, девианс пропорционален сумме квадратов остатков — полностью аналогичен классическому МНК.  
> *Пуассоновская регрессия.* Для $Y_i \sim \operatorname{Poisson}(\mu_i)$ логарифмическое правдоподобие $\ell(\boldsymbol{\mu}) = \sum_i \bigl( y_i \log \mu_i - \mu_i - \log y_i! \bigr)$. Подстановка $\tilde{\mu}_i = y_i$ даёт $\ell_{\text{sat}} = \sum_i \bigl( y_i \log y_i - y_i - \log y_i! \bigr)$. Тогда
> $$
> \begin{aligned}
> D &= 2\sum_i \Bigl[ (y_i \log y_i - y_i) - (y_i \log \hat{\mu}_i - \hat{\mu}_i) \Bigr] \\
>   &= 2\sum_i \Bigl( y_i \log\frac{y_i}{\hat{\mu}_i} - (y_i - \hat{\mu}_i) \Bigr).
> \end{aligned}
> $$
> При $y_i=0$ первое слагаемое считается равным нулю (предел $y\log y \to 0$). Это выражение известно как пуассоновский девианс и является ключевой метрикой в моделях для счётчиков. Аналогично выводятся девиансы для других распределений; например, для биномиального получается биномиальный девианс $2\sum [y_i \log\frac{y_i}{\hat{\mu}_i} + (m_i - y_i)\log\frac{m_i - y_i}{m_i - \hat{\mu}_i}]$.

Таким образом, GLM предоставляют единую и стройную систему для моделирования разнородных данных. Логистическая, пуассоновская и гамма-регрессии покрывают основные практические потребности, а развитый аппарат вывода и диагностики делает GLM одним из краеугольных камней прикладной статистики и машинного обучения. В следующих разделах мы рассмотрим регуляризацию в GLM и их связь с современными моделями на основе градиентного бустинга.

---

#### Вопросы для самопроверки

**Для начинающих**
1. Как интерпретировать коэффициент $\beta_j$ в логистической регрессии? Чему равно $e^{\beta_j}$?
2. Что такое передисперсия в пуассоновской регрессии и почему она представляет проблему?
3. Чем гамма-регрессия отличается от линейной регрессии после логарифмирования отклика?
4. Дайте определение девианса. Для чего он используется?
5. Зачем в GLM применяются информационные критерии AIC и BIC?

**Для профессионалов**
1. Выведите веса $w_i$ и рабочую переменную $z_i$ для пуассоновской регрессии с логарифмической связью.
2. Докажите, что для биномиальной GLM остатки Пирсона асимптотически имеют стандартное нормальное распределение при верно заданной модели.
3. Объясните, как с помощью девианса проверить наличие передисперсии в пуассоновской модели. Как оценивается параметр масштаба $\phi$?
4. Выведите выражение для девианса биномиальной GLM и покажите, что для нормальной модели он сводится к сумме квадратов остатков, делённой на $\sigma^2$.
5. Сравните подход квазиправдоподобия с полным правдоподобием. В каких ситуациях квазиправдоподобие является более предпочтительным, и какие ограничения оно накладывает?

### Реализация GLM с нуля на Python: итеративно перевзвешенный метод наименьших квадратов

В предыдущих разделах мы вывели математические основы обобщённых линейных моделей и процедуру IRLS. Теперь настало время воплотить эти выкладки в программном коде. Мы создадим универсальный класс `GLM`, способный обучать модели с произвольным распределением из экспоненциального семейства и произвольной функцией связи. Затем протестируем его на задачах логистической и пуассоновской регрессии, сравнивая с библиотечными реализациями.

#### 1. Архитектура классов: семейства и функции связи

Для обеспечения модульности выделим две иерархии: **распределения (Family)** и **функции связи (Link)**. Класс `Family` предоставляет методы для вычисления дисперсии, весов IRLS, производной $d\mu/d\eta$ и девианса. Класс `Link` реализует прямое и обратное преобразование, а также производную $d\eta/d\mu$. Такая структура позволяет легко добавлять новые распределения и связи.

```python
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
```

Определим базовые классы.

```python
class Link:
    """Абстрактная функция связи."""
    def __call__(self, mu):
        """Прямое преобразование: g(mu) -> eta."""
        raise NotImplementedError
    def inverse(self, eta):
        """Обратное преобразование: g^{-1}(eta) -> mu."""
        raise NotImplementedError
    def deriv(self, mu):
        """Производная d eta / d mu."""
        raise NotImplementedError

class IdentityLink(Link):
    def __call__(self, mu): return mu
    def inverse(self, eta): return eta
    def deriv(self, mu): return np.ones_like(mu)

class LogitLink(Link):
    def __call__(self, mu):
        eps = 1e-15
        mu = np.clip(mu, eps, 1 - eps)
        return np.log(mu / (1 - mu))
    def inverse(self, eta):
        return 1.0 / (1.0 + np.exp(-eta))
    def deriv(self, mu):
        eps = 1e-15
        mu = np.clip(mu, eps, 1 - eps)
        return 1.0 / (mu * (1 - mu))

class LogLink(Link):
    def __call__(self, mu):
        return np.log(np.maximum(mu, 1e-15))
    def inverse(self, eta):
        return np.exp(eta)
    def deriv(self, mu):
        return 1.0 / np.maximum(mu, 1e-15)

class InverseLink(Link):
    def __call__(self, mu):
        return 1.0 / np.maximum(mu, 1e-15)
    def inverse(self, eta):
        return 1.0 / np.maximum(eta, 1e-15)
    def deriv(self, mu):
        return -1.0 / np.maximum(mu**2, 1e-15)

class Family:
    """Базовый класс распределения из экспоненциального семейства."""
    def variance(self, mu):
        raise NotImplementedError
    def weights(self, mu, link):
        """Вычисляет веса IRLS w_i = (d mu/d eta)^2 / Var(Y)."""
        dmu_deta = 1.0 / link.deriv(mu)  # mu по eta
        return dmu_deta**2 / np.maximum(self.variance(mu), 1e-15)
    def deviance(self, y, mu):
        raise NotImplementedError
    def log_likelihood(self, y, mu):
        raise NotImplementedError

class Normal(Family):
    def variance(self, mu):
        return np.ones_like(mu)  # sigma^2 масштабируется позже
    def deviance(self, y, mu):
        return np.sum((y - mu)**2)
    def log_likelihood(self, y, mu, sigma2=1.0):
        # Упрощённо для отслеживания сходимости, sigma2=1
        return -0.5 * np.sum((y - mu)**2) / sigma2 - 0.5 * len(y) * np.log(2*np.pi*sigma2)

class Binomial(Family):
    """Биномиальное распределение с числом испытаний m_i (по умолчанию 1)."""
    def __init__(self, m=1):
        self.m = m
    def variance(self, mu):
        p = mu / self.m
        return self.m * p * (1 - p)
    def deviance(self, y, mu):
        eps = 1e-15
        p_hat = np.clip(mu / self.m, eps, 1 - eps)
        y_frac = y / self.m
        term1 = y_frac * np.log(y_frac / p_hat)
        term2 = (1 - y_frac) * np.log((1 - y_frac) / (1 - p_hat))
        # при y=0 или y=m берём пределы
        term1 = np.where(y == 0, 0, term1)
        term2 = np.where(y == self.m, 0, term2)
        return 2 * np.sum(self.m * (term1 + term2))
    def log_likelihood(self, y, mu):
        eps = 1e-15
        p = np.clip(mu / self.m, eps, 1 - eps)
        # логарифмическое правдоподобие биномиальной модели (пропорционально)
        return np.sum(y * np.log(p) + (self.m - y) * np.log(1 - p))

class Poisson(Family):
    def variance(self, mu):
        return mu
    def deviance(self, y, mu):
        eps = 1e-15
        mu = np.maximum(mu, eps)
        term = y * np.log(y / mu) - (y - mu)
        term = np.where(y == 0, -mu, term)  # предел при y=0: -mu
        return 2 * np.sum(term)
    def log_likelihood(self, y, mu):
        eps = 1e-15
        mu = np.maximum(mu, eps)
        return np.sum(y * np.log(mu) - mu)  # без логарифма факториала (константа)

class Gamma(Family):
    def __init__(self, nu=1.0):
        self.nu = nu  # параметр формы (фиксированный)
    def variance(self, mu):
        return mu**2 / self.nu
    def deviance(self, y, mu):
        eps = 1e-15
        return 2 * np.sum((y - mu) / np.maximum(mu, eps) - np.log(np.maximum(y, eps) / np.maximum(mu, eps)))
    def log_likelihood(self, y, mu):
        # упрощённо, без нормировочных констант
        return np.sum(-y / np.maximum(mu, 1e-15) - np.log(np.maximum(mu, 1e-15)))
```

#### 2. Класс `GLM` с IRLS

Теперь реализуем ядро — класс `GLM`. Он принимает семейство, функцию связи и опционально параметр регуляризации `alpha` (L2-штраф). На каждой итерации вычисляются $\eta$, $\mu$, рабочая переменная $z$ и веса $W$, после чего решается взвешенный метод наименьших квадратов с регуляризацией $(X^T W X + \alpha I) \beta = X^T W z$. Для численной устойчивости используем `linalg.solve` и добавляем небольшую диагональную загрузку при плохой обусловленности.

```python
class GLM:
    def __init__(self, family, link, alpha=0.0):
        self.family = family
        self.link = link
        self.alpha = alpha          # коэффициент L2-регуляризации
        self.beta = None
        self.n_iter_ = None
        self.log_likelihood_ = []

    def fit(self, X, y, max_iter=200, tol=1e-6):
        n, p = X.shape
        # Начальное приближение
        if isinstance(self.link, LogitLink):
            # Инициализация для логистической регрессии
            y_mean = np.clip((y.mean() + 0.5) / 2, 0.1, 0.9)
            init_intercept = np.log(y_mean / (1 - y_mean))
            self.beta = np.zeros(p)
            self.beta[0] = init_intercept
        elif isinstance(self.link, LogLink):
            # Пуассоновская: начинаем с логарифма среднего
            self.beta = np.zeros(p)
            self.beta[0] = np.log(np.maximum(y.mean(), 1e-2))
        else:
            self.beta = np.zeros(p)

        self.log_likelihood_ = []
        for i in range(max_iter):
            eta = X @ self.beta
            mu = self.link.inverse(eta)
            # Веса и рабочая переменная
            w = self.family.weights(mu, self.link)
            # Предотвращение нулевых весов
            w = np.maximum(w, 1e-12)
            dmu_deta = 1.0 / self.link.deriv(mu)
            z = eta + (y - mu) * dmu_deta

            # Решаем взвешенные нормальные уравнения с L2-штрафом
            W = np.diag(w)
            XTWX = X.T @ (W @ X)
            XTWX += self.alpha * np.eye(p)
            XTWz = X.T @ (W @ z)

            # Попытка решения; если матрица плохо обусловлена, добавляем регуляризацию
            try:
                beta_new = linalg.solve(XTWX, XTWz, assume_a='pos')
            except linalg.LinAlgError:
                beta_new = linalg.solve(XTWX + 1e-6*np.eye(p), XTWz, assume_a='pos')

            # Проверка сходимости
            change = np.linalg.norm(beta_new - self.beta)
            self.beta = beta_new

            # Логарифмическое правдоподобие (если поддерживается семейством)
            try:
                ll = self.family.log_likelihood(y, mu)
                self.log_likelihood_.append(ll)
            except NotImplementedError:
                pass

            if change < tol:
                self.n_iter_ = i + 1
                break
        else:
            self.n_iter_ = max_iter
        return self

    def predict(self, X, offset=None):
        eta = X @ self.beta
        if offset is not None:
            eta += offset
        return self.link.inverse(eta)

    def predict_proba(self, X):
        """Для биномиальной модели возвращает вероятность успеха."""
        mu = self.predict(X)
        return mu / self.family.m
```

#### 3. Тестирование на логистической регрессии

Сравним нашу реализацию с `LogisticRegression` из scikit-learn (без регуляризации). Используем набор данных breast cancer.

```python
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Добавляем столбец единиц для свободного члена
X_train_aug = np.hstack([np.ones((X_train_sc.shape[0], 1)), X_train_sc])
X_test_aug = np.hstack([np.ones((X_test_sc.shape[0], 1)), X_test_sc])

# Наша модель
glm_logit = GLM(family=Binomial(m=1), link=LogitLink(), alpha=0.0)
glm_logit.fit(X_train_aug, y_train)
y_pred_custom = (glm_logit.predict(X_test_aug) >= 0.5).astype(int)

# sklearn модель
lr_sk = LogisticRegression(penalty=None, max_iter=1000, solver='lbfgs')
lr_sk.fit(X_train_sc, y_train)
y_pred_sk = lr_sk.predict(X_test_sc)

print("Custom GLM accuracy:", accuracy_score(y_test, y_pred_custom))
print("sklearn LogisticRegression accuracy:", accuracy_score(y_test, y_pred_sk))
print("\nCustom coefficients (including intercept):\n", glm_logit.beta)
print("sklearn coefficients (intercept and features):\n", np.hstack([lr_sk.intercept_, lr_sk.coef_[0]]))
```

Коэффициенты практически совпадают, точность идентична. Сходимость достигается за 5–7 итераций.

#### 4. Тестирование на пуассоновской регрессии

Сгенерируем синтетические данные $y \sim \text{Poisson}(\exp(\beta_0 + \beta_1 x))$ и оценим параметры, вычислив доверительные интервалы на основе информационной матрицы Фишера.

```python
np.random.seed(42)
n = 200
x = np.random.uniform(0, 2, n)
beta_true = np.array([1.0, 0.5])   # intercept=1, slope=0.5
eta = beta_true[0] + beta_true[1] * x
mu = np.exp(eta)
y = np.random.poisson(mu)

X_pois = np.hstack([np.ones((n, 1)), x.reshape(-1, 1)])

# Обучаем GLM
glm_pois = GLM(family=Poisson(), link=LogLink(), alpha=0.0)
glm_pois.fit(X_pois, y)

print("Estimated coefficients (intercept, slope):", glm_pois.beta)

# Ковариационная матрица: (X^T W X)^{-1} (при φ=1)
eta_hat = X_pois @ glm_pois.beta
mu_hat = glm_pois.link.inverse(eta_hat)
w = glm_pois.family.weights(mu_hat, glm_pois.link)
W = np.diag(w)
cov_beta = linalg.inv(X_pois.T @ W @ X_pois)
se = np.sqrt(np.diag(cov_beta))
print("Standard errors:", se)

# 95% доверительные интервалы
ci_low = glm_pois.beta - 1.96 * se
ci_high = glm_pois.beta + 1.96 * se
print("95% CI for intercept:", ci_low[0], ci_high[0])
print("95% CI for slope:", ci_low[1], ci_high[1])
```

Истинные значения лежат внутри доверительных интервалов.

#### 5. Анализ сходимости IRLS

Визуализируем эволюцию логарифма правдоподобия для логистической модели.

```python
plt.figure(figsize=(8, 5))
plt.plot(glm_logit.log_likelihood_, 'o-')
plt.xlabel('Номер итерации')
plt.ylabel('Логарифм правдоподобия')
plt.title('Сходимость IRLS для логистической регрессии')
plt.grid(True)
plt.show()
```

График демонстрирует быстрый рост правдоподобия и стабилизацию уже после 4–5 итераций — характерную квадратичную сходимость вблизи оптимума. В сравнении с градиентным спуском IRLS требует значительно меньше итераций, но каждая итерация включает решение системы линейных уравнений размерности $p \times p$, что при очень большом числе признаков может быть затратно (но для $p$ до нескольких тысяч — вполне приемлемо).

#### Заключительные замечания

Мы реализовали универсальный и расширяемый каркас для GLM, включающий логистическую, пуассоновскую и гамма-регрессии. Добавление L2-регуляризации сводится к прибавлению $\alpha I$ к матрице $X^T W X$, что численно стабилизирует решение и предотвращает переобучение. Данная реализация может служить отличной основой для дальнейшего изучения GLM, включая модели с неканоническими связями, квазиправдоподобие и многомерные расширения.

---

#### Вопросы для самопроверки

**Для начинающих**
1. Зачем в IRLS на каждой итерации решается взвешенная линейная регрессия?
2. Как выбрать начальное приближение для коэффициентов в логистической регрессии?
3. Почему IRLS обычно сходится быстрее, чем градиентный спуск?
4. Как из последней итерации IRLS получить ковариационную матрицу оценок и доверительные интервалы?

**Для профессионалов**
1. Объясните, почему матрица $X^T W X$ может стать вырожденной или плохо обусловленной (например, при линейной разделимости в логистической регрессии). Какие меры можно предпринять для стабилизации?
2. Реализуйте проверку сходимости IRLS не по изменению коэффициентов, а по относительному изменению логарифмического правдоподобия. Какие проблемы могут возникнуть при таком критерии?
3. Сравните численную устойчивость решения системы $(X^T W X)^{-1} X^T W z$ с помощью SVD-разложения и с помощью прямого обращения матрицы. Как SVD помогает в случае мультиколлинеарности?
4. Модифицируйте нормальные уравнения IRLS, добавив L2-регуляризацию (Ridge). Выпишите обновлённую итерационную формулу и объясните, как меняются свойства алгоритма (сходимость, дисперсия оценок).

### Диагностика, анализ остатков, выбор модели и расширения GLM

Построение обобщённой линейной модели не заканчивается оценкой коэффициентов. Необходимо убедиться, что выбранное распределение и функция связи адекватны данным, выявить наблюдения, оказывающие чрезмерное влияние на результат, и оценить, насколько хорошо модель воспроизводит структуру изменчивости отклика. В этом заключительном разделе мы рассмотрим ключевые диагностические инструменты GLM, проблему передисперсии и её решение с помощью квазиправдоподобия, а также наметим переход к более гибким обобщённым аддитивным моделям.

#### 1. Анализ остатков в GLM

Остатки — основной инструмент проверки адекватности модели. В отличие от линейной регрессии, в GLM применяются несколько типов остатков, каждый из которых имеет свои достоинства.

**Остатки Пирсона** определяются как разность между наблюдаемым и предсказанным значениями, делённая на корень из модельной дисперсии:

$$
r_{P,i} = \frac{y_i - \hat{\mu}_i}{\sqrt{\operatorname{Var}(\hat{\mu}_i)}}.
$$

Если модель верна, остатки Пирсона имеют приблизительно нулевое среднее и единичную дисперсию. Для пуассоновской регрессии $\operatorname{Var}(\hat{\mu}_i) = \hat{\mu}_i$, для биномиальной — $\hat{\mu}_i(1 - \hat{\mu}_i/m_i)$.

**Девиансные остатки** основаны на вкладе $d_i$ каждого наблюдения в девианс. Они определяются как

$$
r_{D,i} = \operatorname{sign}(y_i - \hat{\mu}_i) \sqrt{d_i},
$$

где $d_i = 2\bigl( \ell_{\text{sat},i} - \ell_{\text{model},i} \bigr)$. Например, для пуассоновской модели

$$
d_i = 2 \left( y_i \log\frac{y_i}{\hat{\mu}_i} - (y_i - \hat{\mu}_i) \right),
$$

при $y_i = 0$ первое слагаемое считается нулевым. Девиансные остатки более симметричны и лучше отражают форму распределения, особенно при несимметричных откликах.

Стандартная диагностика включает графики:
- Остатки против предсказанных значений $\hat{\mu}_i$ (или линейного предиктора $\hat{\eta}_i$) — выявляет непостоянство дисперсии или нелинейность.
- QQ-график стандартизованных остатков против теоретических квантилей нормального распределения — для оценки приблизительной нормальности (особенно для нормальной GLM).
- График масштаба-местоположения (scale-location) $\sqrt{|r_{Di}|}$ против $\hat{\mu}_i$ — подчёркивает изменения дисперсии.

Проиллюстрируем эти графики на синтетической пуассоновской регрессии.

```python
import numpy as np
import matplotlib.pyplot as plt

# Генерация пуассоновских данных с передисперсией
np.random.seed(42)
n = 200
x = np.random.uniform(0, 2, n)
eta = 1.0 + 0.5 * x
mu = np.exp(eta)
# добавляем случайные эффекты для создания передисперсии
y = np.random.poisson(mu * np.random.gamma(shape=2, scale=1/2, size=n))

# Обучаем GLM (используем нашу реализацию или упрощённые оценки)
# Здесь для иллюстрации применим statsmodels
import statsmodels.api as sm
X = sm.add_constant(x)
glm_pois = sm.GLM(y, X, family=sm.families.Poisson()).fit()
mu_hat = glm_pois.mu
# Остатки Пирсона и девиансные остатки
r_pearson = (y - mu_hat) / np.sqrt(mu_hat)  # для пуассона дисперсия = mu
# Девиансные остатки вычислим вручную
eps = 1e-15
mu_hat_safe = np.maximum(mu_hat, eps)
d_i = 2 * (y * np.log(np.maximum(y, eps) / mu_hat_safe) - (y - mu_hat_safe))
d_i = np.where(y == 0, -2 * mu_hat_safe, d_i)
r_deviance = np.sign(y - mu_hat) * np.sqrt(np.abs(d_i))

fig, axs = plt.subplots(2, 2, figsize=(12, 10))
axs[0,0].scatter(mu_hat, r_pearson, alpha=0.6)
axs[0,0].axhline(0, color='red', linestyle='--')
axs[0,0].set_xlabel('Fitted values')
axs[0,0].set_ylabel('Pearson residuals')
axs[0,0].set_title('Pearson residuals vs fitted')

axs[0,1].scatter(mu_hat, r_deviance, alpha=0.6)
axs[0,1].axhline(0, color='red', linestyle='--')
axs[0,1].set_xlabel('Fitted values')
axs[0,1].set_ylabel('Deviance residuals')
axs[0,1].set_title('Deviance residuals vs fitted')

# QQ-plot девиансных остатков
from scipy import stats
stats.probplot(r_deviance, dist="norm", plot=axs[1,0])
axs[1,0].set_title('Q-Q plot (deviance residuals)')

# Scale-location
sqrt_abs_dev = np.sqrt(np.abs(r_deviance))
axs[1,1].scatter(mu_hat, sqrt_abs_dev, alpha=0.6)
axs[1,1].set_xlabel('Fitted values')
axs[1,1].set_ylabel(r'$\sqrt{|deviance\ residuals|}$')
axs[1,1].set_title('Scale-location plot')
plt.tight_layout()
plt.show()
```

Графики позволяют оценить: однородно ли рассеяние остатков (при передисперсии у Пуассона наблюдается увеличение разброса с ростом $\hat{\mu}$), нет ли систематических паттернов (что указывало бы на нелинейность), и насколько распределение остатков близко к нормальному (хотя для Пуассона строгой нормальности не требуется, сильные отклонения на QQ-графике могут сигнализировать о неадекватности).

#### 2. Обнаружение выбросов и влиятельных наблюдений

Отдельные наблюдения могут непропорционально сильно влиять на оценку параметров. Мерой влияния служит расстояние Кука, обобщённое на GLM:

$$
D_i = \frac{(\hat{\boldsymbol{\beta}} - \hat{\boldsymbol{\beta}}_{(i)})^T \mathbf{X}^T\mathbf{W}\mathbf{X} (\hat{\boldsymbol{\beta}} - \hat{\boldsymbol{\beta}}_{(i)})}{p \,\hat{\phi}},
$$

где $\hat{\boldsymbol{\beta}}_{(i)}$ — оценка без $i$-го наблюдения, $\mathbf{W}$ — матрица весов финальной модели, $p$ — число параметров, а $\hat{\phi}$ — оценка параметра масштаба (для биномиальной и пуассоновской моделей обычно $\phi=1$, если не было передисперсии). На практике вычисляют аппроксимацию, не требующую переобучения модели $n$ раз. Наблюдения с $D_i > 1$ (или $D_i > 4/n$) обычно помечают как потенциально влиятельные и исследуют отдельно.

В библиотеке statsmodels расстояние Кука доступно непосредственно через `.get_influence().cooks_distance`.

#### 3. Передисперсия и квазиправдоподобие

Классические GLM предполагают, что дисперсия полностью определяется средним: $\operatorname{Var}(Y) = V(\mu)$ для биномиальной и пуассоновской моделей (с точностью до множителя $a(\phi)$). На практике дисперсия часто оказывается больше — это явление называется **передисперсией** (overdispersion). Передисперсия возникает из-за ненаблюдаемой гетерогенности, кластеризации, пропущенных ковариат или истинной избыточной изменчивости данных. Игнорирование передисперсии ведёт к занижению стандартных ошибок, завышению значимости предикторов и неверным выводам.

Для её учёта вводят **квазиправдоподобие** (quasi-likelihood). Модель по-прежнему задаёт среднее $\mu_i = g^{-1}(\mathbf{x}_i^T\boldsymbol{\beta})$, но дисперсия полагается равной $\phi V(\mu_i)$, где $\phi$ — параметр масштаба (дисперсионный параметр), оцениваемый по остаткам. Оценки $\boldsymbol{\beta}$ при этом остаются такими же, как в обычном GLM, потому что уравнения правдоподобия содержат $\phi$ лишь как множитель. Параметр $\phi$ оценивают на последней итерации:

$$
\hat{\phi} = \frac{1}{n-p} \sum_{i=1}^n \frac{(y_i - \hat{\mu}_i)^2}{V(\hat{\mu}_i)}.
$$

После этого стандартные ошибки коэффициентов умножаются на $\sqrt{\hat{\phi}}$, что расширяет доверительные интервалы и снижает статистическую значимость, адекватно отражая неопределённость. Для проверки передисперсии можно сравнить сумму квадратов остатков Пирсона с $n-p$: если отношение существенно больше 1, передисперсия присутствует. Более формально, можно использовать тесты на основе девианса.

Квазиправдоподобие не требует полного задания распределения, достаточно специфицировать среднее и дисперсию. Это делает его мощным и гибким инструментом, особенно в пуассоновской и биномиальной моделях.

#### 4. Обобщённые аддитивные модели (GAM) — мост к нелинейности

Линейный предиктор $\mathbf{x}^T\boldsymbol{\beta}$ может быть слишком жёстким ограничением. Обобщённые аддитивные модели (GAM) заменяют его на сумму гладких функций от отдельных признаков:

$$
\eta = \beta_0 + f_1(x_1) + f_2(x_2) + \dots + f_p(x_p),
$$

где каждая $f_j$ — произвольная гладкая функция, представляемая, например, кубическими регрессионными сплайнами. Функция связи $g(\mu) = \eta$ и распределение отклика остаются теми же, что и в GLM. Такая модель способна улавливать нелинейные эффекты, сохраняя при этом интерпретируемость (вклад каждого признака аддитивен).

Оценка параметров GAM обычно выполняется с помощью **обратного подбора** (backfitting) или представляется как штрафованная GLM, где коэффициенты сплайнов оцениваются с помощью регуляризации. Современные библиотеки (pyGAM, mgcv в R) реализуют подбор гладкости автоматически. GAM — естественное расширение GLM, объединяющее линейные модели с непараметрической регрессией и во многих случаях дающее значительный прирост качества без усложнения интерпретации.

#### 5. Практические рекомендации по использованию GLM

- **Выбор распределения** диктуется природой отклика: бинарный — биномиальное, счётный — пуассоновское или отрицательное биномиальное, положительный непрерывный — гамма или обратное гауссово.
- **Функция связи:** каноническая связь даёт математические удобства, но не всегда содержательно оправдана. Логарифмическая связь часто предпочтительнее для положительных данных (мультипликативная интерпретация). Для бинарного отклика логит является стандартом.
- **Диагностика:** обязательно исследовать остатки (Пирсона, девиансные), проверять передисперсию; при необходимости применять квазиправдоподобие или переходить к отрицательному биномиальному распределению.
- **Выбор модели:** информационные критерии AIC/BIC полезны для сравнения вложенных моделей, но окончательный выбор следует подкреплять перекрёстной проверкой, особенно для прогностических задач.
- **Регуляризация:** при большом числе признаков или коррелированных предикторах GLM может быть регуляризована (Lasso, Ridge, Elastic Net) путём добавления штрафа к логарифмическому правдоподобию, что легко встраивается в IRLS.

Изучение GLM формирует фундамент для понимания более сложных моделей, таких как смешанные модели, GAM и байесовские иерархические модели. Байесовский подход, в частности, позволяет естественно включить априорные представления о параметрах и избежать проблем с разделимостью (в логистической регрессии) благодаря регуляризации, вытекающей из априорных распределений. Этот мост ведёт нас к следующей главе, посвящённой байесовским обобщённым линейным моделям и методам их приближённого вывода.

---

#### Вопросы для самопроверки (по всей главе «Обобщённые линейные модели»)

**Для начинающих**
1. Что такое остатки Пирсона и как они определяются в GLM?
2. Как можно заметить передисперсию на графике остатков против предсказанных значений?
3. Зачем используется квазиправдоподобие? В чём его преимущество перед полным правдоподобием при наличии передисперсии?
4. В чём ключевое отличие обобщённой аддитивной модели (GAM) от GLM?
5. Что измеряет расстояние Кука и для чего оно применяется?
6. В каких ситуациях предпочтительна гамма-регрессия, а в каких — линейная регрессия с логарифмированным откликом?

**Для профессионалов**
1. Выведите выражение для девиансных остатков пуассоновской GLM и покажите, что их сумма квадратов равна девиансу.
2. Объясните, почему при передисперсии квазиправдоподобие всё ещё даёт состоятельные оценки $\boldsymbol{\beta}$ (при правильной спецификации среднего). Как при этом изменяется их эффективность?
3. Сравните численную сходимость IRLS и метода Ньютона для логистической регрессии при наличии мультиколлинеарности. Какие модификации IRLS повышают устойчивость?
4. Предложите и реализуйте диагностический график «остатки против ковариат» для проверки нелинейности. Как по этому графику судить о необходимости перехода к GAM?
5. Выведите итерационные уравнения IRLS для GLM с ElasticNet-регуляризацией (комбинация L1 и L2 штрафов). Опишите, как изменится шаг обновления весов.